## 20 newsgroups Dataset
- ???? Text ?? Toy dataset
- 20?? ?? ??? ???? ???? !
- Multiclass classification
- ? 20,000?? news document ??

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
%matplotlib inline

In [ ]:
from sklearn.datasets import fetch_20newsgroups
news = fetch_20newsgroups(subset='all')
news.keys()

In [ ]:
print(news.data[0])

In [ ]:
news.target[0]

In [ ]:
news.target_names

## ?? ??? ?? ???

* ??? ??
* ???(Preprocessing)

    * ???? ?? ?? (Data Cleansing)
    * CountVectorizer & Tf-idfVectorizer

---
    
* Modeling : BernoulliNB, MultinomialNB ??
  * Cross Validation(Kfold ??)
  
---

* Pipeline ??

---

* Assignment Description
     * ? ?? ???? ???? ?? ??? ???? ????
     * ? ???? Traing / Test Dataset?? ??? 5-fold cross validation(5? ???? training / testset?? ??, KV ??)
     * Naive Bayesian Classifier? Count Vector? ???? ?? ??? ?????
         * NB? multinomial? bernuoil ??? ?? ????
     * ??? ??, TF-IDF vector? ??? ?? (??? - tf-idf scikit-learn)

# Dataset
   * 18846?? ???

In [ ]:
news_df = pd.DataFrame({'News' : news.data, 'Target' : news.target})

In [ ]:
news_df.head()

In [ ]:
# Target ??? -> ?? ???(???? ?? ???? ?? ???? ??? ???)
def word_labeling(lst, df):
    for idx, name in enumerate(lst):
        target_data = df['Target']
        for idx_, num_label in enumerate(target_data):
            if num_label == idx:
                df.loc[idx_, 'Target'] = name
    return df
news_df = word_labeling(news['target_names'], news_df)
news_df.head()

* Data Cleansing
    * ??? ??
    * ??? ?? ??
    * ?? ?? ???? ??
    * ?? ?? ?? ?? : ???? ?? split??? join

In [ ]:
def data_cleansing(df):
    delete_email = re.sub(r'\b[\w\+]+@[\w]+.[\w]+.[\w]+.[\w]+\b', ' ', df)
    delete_number = re.sub(r'\b|\d+|\b', ' ',delete_email)
    delete_non_word = re.sub(r'\b[\W]+\b', ' ', delete_number)
    cleaning_result = ' '.join(delete_non_word.split())
    return cleaning_result 

In [ ]:
news_df.loc[:, 'News'] = news_df['News'].apply(data_cleansing)
news_df.head()

# Vectorizer
* CountVectorizer 
  * ?? ?????? ??? ?? ?? ??? ??? ??
* TfidfVectorizer 
    * ??? ?? ??? ????? ?? ?? ??? ????? ???? ??? ?? ?? ?? ??? ????? ?? ???? ???? ??
    * TF(Term Frequency) : ???? ?? ??? ??? ???? ????? ?? ?
    * DF(Document Frequency) : ?? ??? ?? ??? ?
    * IDF(Inverse Document Frequency) ?? ??? ?? ??? ?? ??? ?? ???? ????? ?? ?? ???
        * log(N / (1 + DF))      
            * N : ?? ??? ?
    * TF-IDF = TF * IDF
* CustomizedVectorizer - StemmedCounterVectorizer, StemmedTfidfVectorizer 

In [ ]:
from nltk import stem
stmmer = stem.SnowballStemmer("english")
sentence = 'looking looks looked'
[stmmer.stem(word) for word in sentence.split()]

In [ ]:
stmmer.stem("images"), stmmer.stem("imaging"), stmmer.stem("imagination")  # ??, ?? ?? ??

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import  nltk
enlish_stemmer = nltk.stem.SnowballStemmer("english")

class StemmedCountVectorizer(CountVectorizer): # CountVectorizer? ?? class 
    def build_analyzer(self):
        analyzer = super(StemmedCountVectorizer,self).build_analyzer() # CountVectorizer? analyzer ??
        return lambda doc: (enlish_stemmer.stem(w) for w in analyzer(doc)) # Stemming ? ??? ???? ??

In [ ]:
StemmedCountVectorizer(min_df=1, stop_words="english").fit([sentence]).vocabulary_

In [ ]:
CountVectorizer(min_df=1, stop_words="english").fit([sentence]).vocabulary_

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

enlish_stemmer = nltk.stem.SnowballStemmer("english")
class StemmedTfidfVectorizer(TfidfVectorizer):
    def build_analyzer(self):
        analyzer = super(StemmedTfidfVectorizer,self).build_analyzer()
        return lambda doc: (enlish_stemmer.stem(w) for w in analyzer(doc))

# Modeling
* Pipeline
* Gridsearch
* Cross Validation

In [ ]:
from nltk import ngrams
sentence = 'this is a foo bar sentences and i want to ngramize it'
n = 6
sixgrams = ngrams(sentence.split(), 3)
for grams in sixgrams:
  print (grams)

In [ ]:
from sklearn.base import TransformerMixin, BaseEstimator

# GaussianNB ???? - Dense? ???? ????? ??
# vectorizer? output : CSR form
# CSR form? Dense form?? ???? ?? class
class DenseTransformer(BaseEstimator, TransformerMixin):

    def transform(self, X, y=None, **fit_params):
        return X.todense()

    def fit_transform(self, X, y=None, **fit_params):
        self.fit(X, y, **fit_params)
        return self.transform(X)

    def fit(self, X, y=None, **fit_params):
        return self

## Modeling Plan - Pipeline

Vectorizer
- Tfidf
- Count
- StemmedTfidf
- StemmedCount

Algorithm
- LogisticRegression
- Bernoulli NB
- Multinomial NB
- Gaussian NB

Metrics
- CV 2 times
- Accuracy

In [ ]:
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline

vectorizer = [CountVectorizer(), TfidfVectorizer(), StemmedCountVectorizer(), StemmedTfidfVectorizer()]
# algorithms = [BernoulliNB(), MultinomialNB(), GaussianNB(), LogisticRegression()]
algorithms = [MultinomialNB(), BernoulliNB()]

pipelines  = []


import itertools
for case in list(itertools.product(vectorizer, algorithms)): # ?? ??? ?? ???? ???
    if isinstance(case[1], GaussianNB): # case[1](=????)? GaussianNB instance??
        case = list(case)
        case.insert(1,  DenseTransformer()) # CSR form? Dense? ???? transform ??? ?
        # case? ?? : [vectorizer, DenseTransformer(), GaussianNB()]
    pipelines.append(make_pipeline(*case))
pipelines

In [ ]:
# Vectorizer Common params
ngrams_params = [(1,1),(1,3)]
stopword_params = ["english"]
lowercase_params = [True, False]
max_df_params = np.linspace(0.4, 0.6, num=4)
min_df_params = np.linspace(0.0, 0.1, num=4)

attributes = {"ngram_range":ngrams_params, "max_df":max_df_params,"min_df":min_df_params,
              "lowercase":lowercase_params,"stop_words":stopword_params}
vectorizer_names = ["countvectorizer","tfidfvectorizer","stemmedcountvectorizer","stemmedtfidfvectorizer"]
vectorizer_params_dict = {}

for vect_name in vectorizer_names:
    vectorizer_params_dict[vect_name] = {}
    for key, value in attributes.items():
        param_name = vect_name + "__" + key
        vectorizer_params_dict[vect_name][param_name] =  value

In [ ]:
attributes.items()

In [ ]:
vectorizer_params_dict

In [ ]:
# Algorithms parameters
# algorithm_names = ["bernoullinb","multinomialnb","gaussiannb","logisticregression"]
algorithm_names = ["multinomialnb", "bernoullinb"]

algorithm_params_dict = {}


#'bernoullinb', BernoulliNB(alpha=1.0, binarize=0.0, class_prior=None, fit_prior=True))])
alpha_params = np.linspace(1.0, 2.0, num=5)
for i in range(2):
    algorithm_params_dict[algorithm_names[i]] = {
    algorithm_names[i]+ "__alpha" : alpha_params    
    }
# algorithm_params_dict[algorithm_names[2]] = {}


# LogisticRegression    
# multi_class : str, {?ovr?, ?multinomial?}, default: ?ovr?
# C : float, default: 1.0
# solver : {?newton-cg?, ?lbfgs?, ?liblinear?, ?sag?, ?saga?},
# n_jobs : int, default: 1
# penalty : str, ?l1? or ?l2?, default: ?l2?

# multi_class_params = ["ovr", "multinomial"]
# c_params = [0.1,  5.0, 7.0, 10.0, 15.0, 20.0, 100.0]
# algorithm_params_dict[algorithm_names[1]] = [{
#     "logisticregression__multi_class" : ["multinomial"],
#     "logisticregression__solver" : ["saga"],
#     "logisticregression__penalty" : ["l1"],
#     "logisticregression__C" : c_params
#     # },{
#     # "logisticregression__multi_class" : ["ovr"],
#     # "logisticregression__solver" : ['liblinear'],
#     # "logisticregression__penalty" : ["l2"],
#     # "logisticregression__C" : c_params
#     }
#     ]
algorithm_params_dict

In [ ]:
pipeline_params= []
for case in list(itertools.product(vectorizer_names, algorithm_names)):
    vect_params = vectorizer_params_dict[case[0]].copy()
    algo_params = algorithm_params_dict[case[1]]
    
    # ? params? ??? ???? ?? ???
    if isinstance(algo_params, dict): # algo_params? ??? dict ??? ??
        vect_params.update(algo_params)
        pipeline_params.append(vect_params) # ??? params? pipeline_parms? ??
    else: # algo_params? list[dict] ??? ??
        temp = []
        for param in algo_params:
            vect_params.update(param)
            temp.append(vect_params)
        pipeline_params.append(temp)
pipeline_params

# Learn! Learn!

In [ ]:
from sklearn.preprocessing import LabelEncoder

X_data = news_df.loc[:, 'News'].tolist()[:1000] # ?? ?? - 1000??
y_data = news_df['Target'].tolist()[:1000]
y = LabelEncoder().fit_transform(y_data)
y

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

scoring = ['accuracy']
estimator_results = []
for i, (estimator, params) in enumerate(zip(pipelines,pipeline_params)):
    gs_estimator = GridSearchCV(
            refit="accuracy", estimator=estimator, param_grid=params, scoring=scoring, cv=5, verbose=1, n_jobs=36)
    print(gs_estimator)

    gs_estimator.fit(X_data, y)
    estimator_results.append(gs_estimator)

In [ ]:
import pandas as pd
from pandas import DataFrame
result_df_dict = {}
result_attributes = ["vectorizer", "model", "accuracy", "recall_macro","precision_macro" , "min_df", 
                     "lowercase", "max_df", "binarize", "alpha", "ngram_range"
                     "multi_class", "penalty", "solver", "C"]

pieline_list =  list(itertools.product(vectorizer_names, algorithm_names))

for att in result_attributes:
    result_df_dict[att] = [None for i in range(16)]

result_df = DataFrame(result_df_dict)
result_df

In [ ]:
pieline_list

In [ ]:
for i, estiamtor in enumerate(estimator_results):
    best_estimator = estiamtor.best_estimator_
    best_index = estiamtor.best_index_
    result_df_dict["vectorizer"][i] = pieline_list[i][0]
    result_df_dict["model"][i] = pieline_list[i][1]
    result_df_dict["accuracy"][i] = estiamtor.best_score_
#     result_df_dict["recall_micro"][i] = estiamtor.cv_results_["mean_test_recall_micro"][best_index]
#     result_df_dict["precision_micro"][i] = estiamtor.cv_results_["mean_test_precision_micro"][best_index]
    for key, value in estiamtor.best_params_.items():
        if key.split("__")[1] in result_df_dict:
            name = key.split("__")[1]
            result_df_dict[name][i] = value
#     print(estiamtor.best_params_)
#     print(a.named_steps)

In [ ]:
result_df = DataFrame(result_df_dict, columns=result_attributes)
result_df.sort_values("accuracy",ascending=False)